In [1]:
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import accelerate
from transformers import pipeline

/home/ltnga/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
file_path = "/home/ltnga/NguyenTrinhTest/test.json"
with open(file_path, 'r', encoding='utf-8') as f:
        data =json.load(f)

In [3]:
print(data)

[{'text': [{'text': 'Mười vấn đề phụ khoa thường gặp nhất phụ nữ cần biết (Phần 1)\nBài viết thứ 00/52 thuộc chủ đề “Các vấn đề Phụ khoa”\n\nHiệu đính: BS. Trương Khánh Hùng\n\nRối loạn phụ khoa là gì?\nRối loạn phụ khoa là những rối loạn ảnh hưởng đến hệ sinh dục ở phụ nữ. Các cơ quan trong hệ sinh dục nữ bao gồm vú, tử cung, ống dẫn trứng, buồng trứng và cơ quan sinh dục ngoài.\n\nTrong cuộc đời mỗi người phụ nữ, đều có thời điểm sẽ mắc một số bệnh phụ khoa nào đó. Những rối loạn phụ khoa này ảnh hưởng rất lớn đến chức năng tình dục. Những rối loạn này không nên xem nhẹ vì chúng có thể ảnh hưởng xấu đến khả năng sinh sản, hoặc tệ hơn có thể đe dọa đến tính mạng của phụ nữ.\n\nMười rối loạn phụ khoa phổ biến nhất ở phụ nữ là gì?\nBài viết này sẽ đề cập đến mười rối loạn phụ khoa hàng đầu phổ biến nhất, bao gồm:\n\nĐau bụng kinh hoặc đau nhiều khi hành kinh\nKhí hư (ra nhiều nhiều huyết trắng âm đạo)\nVô kinh hoặc không có kinh trong một thời gian dài\nHội chứng buồng trứng đa nang (PC

In [4]:
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16
    )

Loading checkpoint shards: 100%|██████████| 8/8 [10:30<00:00, 78.85s/it]


In [5]:
import json
import re
from rapidfuzz import fuzz

def is_similar(q1: str, q2: str, threshold=70) -> bool:
    """
    Sử dụng fuzzy matching để so sánh hai câu hỏi.
    Nếu điểm tương đồng >= threshold, coi như chúng trùng ý.
    """
    ratio = fuzz.ratio(q1.lower(), q2.lower())
    return ratio >= threshold


def extract_qa_pairs(generated_text: str):
    """
    Tách tất cả cặp hỏi-đáp từ một đoạn văn bản đầu ra của mô hình.
    Trả về danh sách dict có keys: 'question' và 'answer'.
    """
    pattern = r"Câu hỏi:\s*(.*?)\s*Trả lời:\s*(.*?)(?=\s*Câu hỏi:|$)"
    matches = re.findall(pattern, generated_text, flags=re.DOTALL)

    results = []
    for q_content, a_content in matches:
        q_clean = clean_text(q_content)
        a_clean = clean_text(a_content)
        results.append({
            "question": q_clean,
            "answer": a_clean
        })
    return results


def clean_text(text: str) -> str:
    """
    Loại bỏ các ký tự không mong muốn như '---', '###', dấu xuống dòng thừa, 
    khoảng trắng thừa ở đầu/cuối, v.v...
    """
    text = re.sub(r"---+", "", text)   
    text = re.sub(r"#+", "", text)     
    text = re.sub(r"\s*\n\s*", " ", text) 
    text = text.strip()  
    return text
prompt_template = """
Bạn là một chuyên gia phân tích y khoa chuyên sâu về sản, phụ khoa. Nhiệm vụ của bạn là tạo các cặp câu hỏi và trả lời đầy đủ, chính xác dựa trên đoạn văn bản sau đây:

---

### Văn bản cung cấp:
{context}

---

### Quy tắc tạo **Câu hỏi**:
1. **Loại câu hỏi cần tạo**:
   - Tập trung vào các thông tin quan trọng trong lĩnh vực sản, phụ khoa.
   - Dùng ngôn ngữ dễ hiểu, ngắn gọn nhưng rõ ràng.
   - Định nghĩa thuật ngữ y khoa hoặc bệnh lý liên quan đến phụ khoa và sản.
   - Nguyên nhân, triệu chứng, hoặc dấu hiệu lâm sàng của các rối loạn phụ khoa.
   - Cách lây nhiễm, chẩn đoán, điều trị, hoặc chăm sóc liên quan đến sản, phụ khoa.
   - Biện pháp phòng ngừa, cải thiện sức khỏe phụ nữ trong lĩnh vực sản, phụ khoa.
   - Các thông tin bổ sung như chi phí, cơ sở y tế phù hợp trong lĩnh vực này.

2. **Yêu cầu nội dung câu hỏi**:
   - Tập trung vào những thông tin quan trọng và có tính ứng dụng cao.
   - Sử dụng ngôn ngữ rõ ràng, dễ hiểu, phù hợp với người dùng phổ thông.
   - Không vượt quá 20 từ cho mỗi câu hỏi.
   - **Không được trùng lặp nội dung của các câu hỏi đã tạo.**

---

### Quy tắc tạo **Câu trả lời**:
1. **Nội dung trả lời**:
   - Cung cấp thông tin đầy đủ, rõ ràng dựa trên câu hỏi.
   - Mỗi câu trả lời không vượt quá 50 từ.
   - Mỗi cặp câu hỏi phải khác nhau, không được trùng lặp.

2. **Hình thức trình bày**:
   - Trình bày hệ thống, dễ hiểu, không rườm rà.
   - Duy trì phong cách chuyên nghiệp, thân thiện và dễ tiếp cận.

---

### Định dạng yêu cầu:
- **Câu hỏi:** Bắt đầu bằng "Câu hỏi: ".
- **Câu trả lời:** Bắt đầu bằng "Trả lời: ".
- Mỗi cặp câu hỏi-trả lời được phân tách bằng một dòng trống.

---

### Ví dụ mẫu:
Câu hỏi: U xơ tử cung là gì và ảnh hưởng như thế nào đến sức khỏe phụ nữ?

Trả lời: U xơ tử cung là khối u lành tính, gây đau và rối loạn kinh nguyệt, có thể ảnh hưởng đến khả năng sinh sản.

---

Bây giờ, hãy tạo **nhiều cặp câu hỏi và câu trả lời** (nhiều  nhất 10 cặp) dựa trên văn bản đã cung cấp. Đảm bảo rằng mỗi câu trả lời không chứa nội dung của câu hỏi tiếp theo và không có câu hỏi nào trùng nhau.
"""

# Đọc file JSON
file_path = "/home/ltnga/NguyenTrinhTest/test.json"
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

qa_pairs = []
existing_questions = [] 
num_qa_pairs = 10        
for item in data:
    # Nếu item['text'] là list, kiểm tra từng phần tử
    if isinstance(item['text'], list):
        # Nếu phần tử là chuỗi thì join trực tiếp, nếu là dict thì lấy giá trị của key 'text'
        texts = []
        for elem in item['text']:
            if isinstance(elem, str):
                texts.append(elem)
            elif isinstance(elem, dict) and 'text' in elem:
                texts.append(elem['text'])
            else:
                texts.append(str(elem))
        text = "\n\n".join(texts)
    else:
        text = item['text']
    
    paragraphs = text.split("\n\n")
    
    for paragraph in paragraphs:
        for _ in range(num_qa_pairs):
            prompt = prompt_template.format(context=paragraph)

            # Sinh output từ model
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            outputs = model.generate(
                inputs.input_ids,
                max_new_tokens=516,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            response = tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Tách phần output: bỏ prompt, lấy phần mô hình sinh
            generated_content = response.split(prompt, 1)[-1].strip()

            # Tách cặp Hỏi-Đáp
            pairs = extract_qa_pairs(generated_content)

            if not pairs:
                # Nếu không có cặp Q&A nào
                qa_pairs.append({
                    "question": "",
                    "answer": "",
                    "raw": generated_content
                })
            else:
                # Kiểm tra trùng lặp bằng fuzzy matching
                for pair in pairs:
                    q_current = pair['question']
                    is_duplicate = any(is_similar(q_current, q_exist) for q_exist in existing_questions)
                    
                    if not is_duplicate:
                        existing_questions.append(q_current)
                        qa_pairs.append(pair)



output_file = "qa-test2.js"
with open(output_file, "w", encoding="utf-8") as f:
    f.write("const qaPairs = " + json.dumps(qa_pairs, ensure_ascii=False, indent=4) + ";")

print(f"Đã lưu {len(qa_pairs)} cặp Q&A (không trùng/ gần giống) vào file {output_file}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


KeyboardInterrupt: 

In [ ]:
import time
from openai import OpenAI

# Định nghĩa prompt template
prompt_template = """
Bạn là một chuyên gia phân tích y khoa chuyên sâu về sản, phụ khoa. Nhiệm vụ của bạn là tạo các cặp câu hỏi và trả lời đầy đủ, chính xác dựa trên đoạn văn bản sau đây:

---

### Văn bản cung cấp:
{context}

---

### Quy tắc tạo **Câu hỏi**:
1. **Loại câu hỏi cần tạo**:
   - Tập trung vào các thông tin quan trọng trong lĩnh vực sản, phụ khoa.
   - Dùng ngôn ngữ dễ hiểu, ngắn gọn nhưng rõ ràng.
   - Định nghĩa thuật ngữ y khoa hoặc bệnh lý liên quan đến phụ khoa và sản.
   - Nguyên nhân, triệu chứng, hoặc dấu hiệu lâm sàng của các rối loạn phụ khoa.
   - Cách lây nhiễm, chẩn đoán, điều trị, hoặc chăm sóc liên quan đến sản, phụ khoa.
   - Biện pháp phòng ngừa, cải thiện sức khỏe phụ nữ trong lĩnh vực sản, phụ khoa.
   - Các thông tin bổ sung như chi phí, cơ sở y tế phù hợp trong lĩnh vực này.

2. **Yêu cầu nội dung câu hỏi**:
   - Tập trung vào những thông tin quan trọng và có tính ứng dụng cao.
   - Sử dụng ngôn ngữ rõ ràng, dễ hiểu, phù hợp với người dùng phổ thông.
   - Không vượt quá 20 từ cho mỗi câu hỏi.
   - **Không được trùng lặp nội dung của các câu hỏi đã tạo.**

---

### Quy tắc tạo **Câu trả lời**:
1. **Nội dung trả lời**:
   - Cung cấp thông tin đầy đủ, rõ ràng dựa trên câu hỏi.
   - Mỗi câu trả lời không vượt quá 50 từ.
   - Mỗi cặp câu hỏi phải khác nhau, không được trùng lặp.

2. **Hình thức trình bày**:
   - Trình bày hệ thống, dễ hiểu, không rườm rà.
   - Duy trì phong cách chuyên nghiệp, thân thiện và dễ tiếp cận.

---

### Định dạng yêu cầu:
- **Câu hỏi:** Bắt đầu bằng "Câu hỏi: ".
- **Câu trả lời:** Bắt đầu bằng "Trả lời: ".
- Mỗi cặp câu hỏi-trả lời được phân tách bằng một dòng trống.

---

### Ví dụ mẫu:
Câu hỏi: U xơ tử cung là gì và ảnh hưởng như thế nào đến sức khỏe phụ nữ?

Trả lời: U xơ tử cung là khối u lành tính, gây đau và rối loạn kinh nguyệt, có thể ảnh hưởng đến khả năng sinh sản.

---

Bây giờ, hãy tạo **nhiều cặp câu hỏi và câu trả lời** (nhiều nhất 10 cặp) dựa trên văn bản đã cung cấp. Đảm bảo rằng mỗi câu trả lời không chứa nội dung của câu hỏi tiếp theo và không có câu hỏi nào trùng nhau.
"""

# Dữ liệu mẫu của bạn:
context = """
Điều trị vô sinh


Bài viết thứ 00/52 thuộc chủ đề “Các vấn đề Phụ khoa”

Nội dung chính  Ẩn 
1 Vô sinh là gì?
2 Các phương pháp điều trị vô sinh?
2.1 Những thay đổi về lối sống có thể làm tăng khả năng thụ thai
2.2 Phẫu thuật được sử dụng như thế nào để điều trị vô sinh ở phụ nữ?
2.3 Phẫu thuật được sử dụng như thế nào để điều trị vô sinh ở nam giới?
2.4 Những vấn đề về hormone ở phụ nữ được điều trị như thế nào?
3 Thế nào là gây rụng trứng?
3.1 Gây rụng trứng được thực hiện như thế nào?
3.2 Những thuốc nào ngoài Clomiphene Citrate được sử dụng để gây rụng trứng?
3.3 Gonadotropin được sử dụng như thế nào?
3.4 Những nguy cơ có liên quan đến phương pháp gây rụng trứng?
4 Thế nào là bơm tinh trùng vào buồng tử cung?
4.1 Những nguy cơ của phương pháp bơm tinh trùng vào buồng tử cung?
5 Kỹ thuật hỗ trợ sinh sản (ART) là gì?
5.1 Tỉ lệ thành công của ART?
6 Thụ tinh trong ống nghiệm
6.1 Thụ tinh trong ống nghiệm được thực hiện như thế nào?
6.2 Thế nào là tiêm tinh trùng vào bào tương trứng (ICSI)?
6.3 Những nguy cơ nào liên quan đến IVF?
6.4 Những bước nào có thể thực hiện để giúp ngăn ngừa đa thai khi thực hiện IVF?
Vô sinh là gì?
Vô sinh được định nghĩa là không mang thai sau 1 năm có quan hệ tình dục đều đặn mà không sử dụng biện pháp tránh thai (xem bài Khám vô sinh). Vô sinh có thể do một số yếu tố gây ra. Các yếu tố từ người đàn ông hay phụ nữ đều có thể góp phần gây vô sinh.

Các phương pháp điều trị vô sinh?
Các phương pháp điều trị vô sinh phụ thuộc vào nguyên nhân gây vô sinh. Những thay đổi về lối sống, thuốc, phẫu thuật hay kỹ thuật hỗ trợ sinh sản (ART) có thể được đề xuất. Nhiều cách điều trị khác nhau có thể phối hợp để nâng cao kết quả. Vô sinh thường có thể điều trị thành công ngay cả khi không tìm được nguyên nhân nào.

Những thay đổi về lối sống có thể làm tăng khả năng thụ thai
Nếu các yếu tố liên quan đến lối sống được phát hiện, bạn có thể sẽ cần phải giảm hoặc tăng cân, tập thể dục nhiều hơn hoặc ít hơn. Bạn và bạn tình có thể cần giảm sử dụng chất có cồn, bỏ thuốc lá hay ngừng sử dụng các loại thuốc trái phép.

Phẫu thuật được sử dụng như thế nào để điều trị vô sinh ở phụ nữ?
Ở nữ giới, phẫu thuật có thể điều trị các ống dẫn trứng bị tắc nghẽn hay tổn thương. Phẫu thuật được sử dụng để điều trị lạc nội mạc tử cung, thường liên quan đến vô sinh (xem bài Lạc nội mạc tử cung).

Phẫu thuật được sử dụng như thế nào để điều trị vô sinh ở nam giới?
Ở nam giới, phẫu thuật có thể được sử dụng để điều trị một vài vấn đề gây vô sinh. Một vấn đề phổ biến dẫn đến tình trạng vô sinh ở nam giới là giãn tĩnh mạch thừng tinh, đôi khi có thể điều trị bằng phẫu thuật.

Những vấn đề về hormone ở phụ nữ được điều trị như thế nào?
Nồng độ bất thường của các hormone có thể gây ra rụng trứng không đều hoặc không rụng trứng. Ví dụ, hội chứng buồng trứng đa nang là một tình trạng trong đó một số hormone có nồng độ bất thường và chu kỳ kinh nguyệt không đều đặn hay không có. Đó là một nguyên nhân thường gặp của vô sinh. Tình trạng này thường được điều trị bằng cách thay đổi về lối sống hay dùng thuốc. Progesterone có thể được sử dụng để điều trị một vài vấn đề về rụng trứng. Các rối loạn về hormone khác có thể ảnh hưởng đến khả năng sinh sản ở phụ nữ như bệnh tuyến giáp nên được kiểm tra và loại trừ.

Thế nào là gây rụng trứng?
Gây rụng trứng là việc sử dụng các thuốc để làm cho buồng trứng của người phụ nữ phóng thích ra một trứng. Phương pháp này được sử dụng khi sự rụng trứng không đều đặn hoặc hoàn toàn không xảy ra và các nguyên nhân khác đã được loại trừ. Tuy nhiên, một số trường hợp thuốc sử dụng lại gây rụng nhiều trứng hơn mong đợi, gọi là đa phóng noãn.

Gây rụng trứng được thực hiện như thế nào?
Thuốc hay được dùng để gây rụng trứng nhất là Clomiphene Citrate. Khoảng 40% phụ nữ thụ thai nhờ sử dụng thuốc này trong vòng sáu chu kỳ kinh nguyệt. Các tác dụng phụ thường nhẹ, bao gồm những cơn bốc hỏa, ngực căng đau, buồn nôn và thay đổi tâm trạng thất thường.

Những thuốc nào ngoài Clomiphene Citrate được sử dụng để gây rụng trứng?
Nếu Clomiphene Citrate không thành công, các loại thuốc được gọi là Gonadotropin có thể được sử dụng để gây rụng trứng. Gonadotropin còn được sử dụng khi cần nhiều trứng cho ART hoặc các phương pháp điều trị vô sinh khác. Đây gọi là gây rụng nhiều trứng hay đa phóng noãn.

Gonadotropin được sử dụng như thế nào?
Gonadotropin được cho vào giai đoạn đầu của chu kỳ kinh nguyệt. Xét nghiệm máu và siêu âm được sử dụng để theo dõi sự phát triển và trưởng thành của các nang noãn (các túi nhỏ chứa trứng bên trong). Khi kết quả của các xét nghiệm này cho thấy các nang noãn đã đạt được kích thước nhất định, một thuốc khác gọi là human chorionic gonadotropin (hCG) có thể được cho sử dụng. Thuốc này có tác dụng kích thích rụng trứng.

Những nguy cơ có liên quan đến phương pháp gây rụng trứng?
Sinh đôi xảy ra ở 10% phụ nữ được điều trị với Clomiphene Citrate, sinh ba hoặc nhiều hơn hiếm gặp. Nguy cơ đa thai là cao hơn khi Gonadotropin được sử dụng, có đến 30% trường hợp thụ thai với Gonadotropin là đa thai. Khoảng 2/3 những trường hợp này là sinh đôi và 1/3 là sinh ba hoặc hơn.

Gây rụng trứng có thể dẫn đến hội chứng quá kích buồng trứng. Phần lớn các trường hợp này là nhẹ. Với các trường hợp nghiêm trọng, việc nhập viện là cần thiết.

Thế nào là bơm tinh trùng vào buồng tử cung?
Với phương pháp bơm tinh trùng vào buồng tử cung, một lượng lớn tinh trùng khỏe mạnh được đưa vào tử cung vào gần thời điểm rụng trứng. Phương pháp này thường được sử dụng cùng với gây rụng trứng hoặc gây rụng nhiều trứng. Bạn tình của người phụ nữ hoặc một người hiến tặng sẽ cung cấp tinh trùng. Tinh trùng đã được thu nhận trước đó và trữ lạnh cũng có thể được sử dụng.

Những nguy cơ của phương pháp bơm tinh trùng vào buồng tử cung?
Nếu các thuốc gây rụng trứng được sử dụng với phương pháp bơm tinh trùng vào buồng tử cung, đa thai có thể xảy ra. Nếu có quá nhiều trứng phát triển trong chu kỳ điều trị, việc bơm tinh trùng có thể bị hủy bỏ để tránh nguy cơ đa thai.

Xem thêm bài viết Bài 11: Bơm tinh trùng vào buồng tử cung của BS. Lê Tiểu My
Kỹ thuật hỗ trợ sinh sản (ART) là gì?
ART bao gồm tất cả các phương pháp điều trị vô sinh trong đó trứng và tinh trùng được lấy ra ngoài cơ thể và thao tác trong phòng lab để cho thụ tinh với nhau. ART thường bao gồm thụ tinh trong ống nghiệm (IVF). Trong IVF, tinh trùng được kết hợp với trứng trong phòng lab để tạo thành phôi và phôi sẽ được chuyển vào trong tử cung. IVF được thực hiện trong các trường hợp vô sinh do những nguyên nhân sau:

Ống dẫn trứng bị tổn thương hay tắc nghẽn mà không điều trị được bằng phẫu thuật
Một số yếu tố gây vô sinh ở nam giới
Lạc nội mạc tử cung nặng
Suy buồng trứng sớm
Vô sinh không rõ nguyên nhân
Tỉ lệ thành công của ART?
Các Trung tâm Kiểm soát và Phòng bệnh sẽ đưa thông tin này lên website của họ (Centers for Disease Control and Prevention). Tỉ lệ thành công cũng được nêu ra trên website của Hội Kỹ thuật Hỗ trợ Sinh sản (SART).

Thụ tinh trong ống nghiệm
Thụ tinh trong ống nghiệm được thực hiện như thế nào?
IVF được tiến hành theo các chu kỳ. Có thể phải mất hơn một chu kỳ để có được thành công. Tinh trùng được lấy từ bạn tình của bạn hoặc từ một người hiến tặng. Tinh trùng có thể được thu nhận và trữ lạnh để sử dụng sau đó với IVF. Sự rụng trứng thường được kích thích bởi Gonadotropin do đó sẽ có nhiều trứng phát triển. Trứng cũng có thể được lấy từ một người hiến tặng. Trứng đã được trữ lạnh trước đó cũng có thể được sử dụng.

Các trứng  sau khi trưởng thành sẽ được lấy ra khỏi buồng trứng. Tinh trùng khỏe mạnh được tiêm vào trứng trong phòng lab. Các trứng này được kiểm tra vào ngày tiếp theo để xem chúng đã được thụ tinh hay chưa. Một vài ngày sau, một hay một vài phôi sẽ được đưa vào tử cung của bạn. Phôi có thể nhận từ một người hiến tặng. Những phôi khỏe mạnh nếu không được đưa vào tử cung sẽ được trữ lạnh và bảo quản để sử dụng sau.

Thế nào là tiêm tinh trùng vào bào tương trứng (ICSI)?
Thỉnh thoảng, một tinh trùng duy nhất sẽ được tiêm vào mỗi trứng. Phương pháp này gọi là ICSI. ICSI được đề nghị nếu tinh trùng của bạn tình của bạn bị bất thường. Trong ICSI, chỉ cần duy nhất một tinh trùng khỏe mạnh cho mỗi trứng. Vài ngày sau đó, một hoặc một vài phôi sẽ được đưa vào tử cung qua đường âm đạo.

Những nguy cơ nào liên quan đến IVF?
Khi thực hiện IVF, nguy cơ đa thai sẽ cao hơn. IVF còn liên quan đến nguy cơ tăng các dị tật bẩm sinh. Những dị tật này bao gồm chẻ vòm, các vấn đề về tim và các vấn đề với ống tiêu hóa. Tuy nhiên, về tổng thể, sự tăng nguy cơ dị tật bẩm sinh là khá nhỏ.

Những bước nào có thể thực hiện để giúp ngăn ngừa đa thai khi thực hiện IVF?
Nhiều việc có thể thực hiện để giúp ngăn ngừa đa thai. Nếu kết quả của các xét nghiệm cho thấy quá nhiều trứng đang phát triển, liều hCG dùng để kích thích rụng trứng có thể bị hoãn hay hủy. Bác sỹ cũng có thể giới hạn số lượng phôi đưa vào tử cung của bạn.

Chú giải

Kỹ thuật hỗ trợ sinh sản (ART): Một nhóm các phương pháp điều trị vô sinh trong đó một trứng được thụ tinh với một tinh trùng ở ngoài cơ thể; trứng đã thụ tinh sau đó sẽ được chuyển vào tử cung.
Trứng: tế bào sinh sản ở nữ được sản xuất và phóng thích từ buồng trứng; còn được gọi là noãn.
Phôi: cá thể đang phát triển từ thời điểm bắt đầu hình thành trong tử cung đến hết 8 tuần đầu của thai kỳ.
Lạc nội mạc tử cung: Tình trạng trong đó những mô tương tự với lớp lót bình thường của tử cung được tìm thấy bên ngoài tử cung, thường là ở buồng trứng, ống dẫn trứng và các cấu trúc khác trong vùng chậu.
Ống dẫn trứng: Các ống dẫn nơi trứng di chuyển từ buồng trứng đến tử cung.
Nang noãn: Các cấu trúc dạng túi nơi trứng phát triển bên trong buồng trứng.
Hormone: Chất được sản xuất bởi cơ thể để điều khiển chức năng của các cơ quan khác nhau.
Human Chorionic Gonadotropin (hCG): Một hormone được sản xuất trong thai kỳ, sự có mặt của chất này là căn bản cho hầu hết các xét nghiệm thử thai. Ngoài ra, hCG còn có thành phần giống hormone LH, nên còn được sử dụng thay thế LH để kích thích rụng trứng.
Thụ tinh trong ống nghiệm (IVF): Một thủ thuật trong đó một trứng được lấy ra từ buồng trứng của người phụ nữ, thụ tinh trên đĩa trong phòng thí nghiệm với một tinh trùng của người đàn ông và sau đó được đưa trở lại vào tử cung của người phụ nữ để bắt đầu một thai kỳ.
Đa thai: Sự mang thai trong đó có hai hợp tử hoặc hơn.
Hội chứng quá kích buồng trứng: Một tình trạng gây ra bởi sự đáp ứng kích thích quá mức của các buồng trứng, dẫn đến các buồng trứng sưng đau và có dịch trong ổ bụng và phổi.
Buồng trứng: Hai tuyến, nằm hai bên của tử cung, chứa các trứng để phóng thích khi rụng trứng và sản xuất ra các hormone.
Sự rụng trứng: Sự phóng thích một trứng từ một trong hai buồng trứng.
Hội chứng buồng trứng đa nang: Một tình trạng được xác định bởi 2 trong 3 đặc điểm sau: sự hiện diện của rất nhiều nang trứng trong các buồng trứng, chu kỳ kinh nguyệt không đều và sự tăng nồng độ của các hormone nhất định.
Suy buồng trứng sớm: Một tình trạng trong đó sự rụng trứng và chu kỳ kinh nguyệt chấm dứt trước năm 35 tuổi.
Progesterone: Một hormone nữ được sản xuất ở các buồng trứng và giúp chuẩn bị lớp lót của tử cung cho sự mang thai.
Quan hệ tình dục: Hành vi trong đó dương vật của nam được đưa vào âm đạo của nữ (còn gọi là “giao hợp” hay “làm tình”).
Tinh trùng: Một tế bào của đàn ông được sản xuất trong các tinh hoàn và có thể thụ tinh với một trứng của phụ nữ.
Siêu âm: Một xét nghiệm trong đó sóng siêu âm được sử dụng để kiểm tra các cấu trúc bên trong.
Tử cung: Một cơ quan có thành phần chính là cơ, nằm trong vùng chậu của nữ, chứa đựng và nuôi dưỡng cho bào thai đang phát triển trong suốt quá trình mang thai.
Giãn tĩnh mạch thừng tinh: Sự giãn các tĩnh mạch ở bìu.
"""

# Tạo prompt bằng cách chèn dữ liệu vào template
prompt = prompt_template.format(context=context)

# Khởi tạo client với API Key và Base URL của DeepSeek
client = OpenAI(api_key="dskey", base_url="https://api.deepseek.com")

# Gọi API để tạo cặp Q&A dựa trên prompt
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "You are a helpful assistant that creates Q&A pairs from content."},
        {"role": "user", "content": prompt}
    ],
    stream=False
)

# In ra nội dung Q&A được tạo ra
print(response.choices[0].message.content)


Câu hỏi: Vô sinh là gì và được định nghĩa như thế nào?

Trả lời: Vô sinh là tình trạng không mang thai sau 1 năm quan hệ tình dục đều đặn mà không sử dụng biện pháp tránh thai, có thể do nguyên nhân từ cả nam và nữ.

Câu hỏi: Những thay đổi lối sống nào giúp tăng khả năng thụ thai?

Trả lời: Giảm hoặc tăng cân, điều chỉnh mức độ tập thể dục, giảm rượu bia, bỏ thuốc lá và ngừng sử dụng thuốc trái phép có thể giúp tăng khả năng thụ thai.

Câu hỏi: Phẫu thuật được sử dụng như thế nào để điều trị vô sinh ở phụ nữ?

Trả lời: Phẫu thuật giúp điều trị tắc nghẽn ống dẫn trứng hoặc lạc nội mạc tử cung, những nguyên nhân phổ biến gây vô sinh ở phụ nữ.

Câu hỏi: Phẫu thuật có thể điều trị vô sinh ở nam giới như thế nào?

Trả lời: Phẫu thuật có thể điều trị giãn tĩnh mạch thừng tinh, một nguyên nhân phổ biến gây vô sinh ở nam giới.

Câu hỏi: Gây rụng trứng là gì và được thực hiện như thế nào?

Trả lời: Gây rụng trứng là sử dụng thuốc để kích thích buồng trứng phóng thích trứng, thường dùng Clomiph

In [ ]:
import time
import pandas as pd
from openai import OpenAI

# Đọc file Excel chứa các cột 'content' và 'link'
df = pd.read_excel("Pregnancy_data.xlsx")  # Thay đổi đường dẫn file nếu cần
# Lấy 5 bài viết đầu tiên để test
df_test = df

# Định nghĩa prompt template
prompt_template = """
Bạn là một chuyên gia phân tích y khoa chuyên sâu về sản, phụ khoa. Nhiệm vụ của bạn là tạo các cặp câu hỏi và trả lời đầy đủ, chính xác dựa trên đoạn văn bản sau đây:

---

### Văn bản cung cấp:
{context}

---

### Quy tắc tạo **Câu hỏi**:
1. **Loại câu hỏi cần tạo**:
   - Tập trung vào các thông tin quan trọng trong lĩnh vực sản, phụ khoa.
   - Dùng ngôn ngữ dễ hiểu, ngắn gọn nhưng rõ ràng.
   - Định nghĩa thuật ngữ y khoa hoặc bệnh lý liên quan đến phụ khoa và sản.
   - Nguyên nhân, triệu chứng, hoặc dấu hiệu lâm sàng của các rối loạn phụ khoa.
   - Cách lây nhiễm, chẩn đoán, điều trị, hoặc chăm sóc liên quan đến sản, phụ khoa.
   - Biện pháp phòng ngừa, cải thiện sức khỏe phụ nữ trong lĩnh vực sản, phụ khoa.
   - Các thông tin bổ sung như chi phí, cơ sở y tế phù hợp trong lĩnh vực này.

2. **Yêu cầu nội dung câu hỏi**:
   - Tập trung vào những thông tin quan trọng và có tính ứng dụng cao.
   - Sử dụng ngôn ngữ rõ ràng, dễ hiểu, phù hợp với người dùng phổ thông.
   - Không vượt quá 20 từ cho mỗi câu hỏi.
   - **Không được trùng lặp nội dung của các câu hỏi đã tạo.**

---

### Quy tắc tạo **Câu trả lời**:
1. **Nội dung trả lời**:
   - Cung cấp thông tin đầy đủ, rõ ràng dựa trên câu hỏi.
   - Mỗi câu trả lời không vượt quá 50 từ.
   - Mỗi cặp câu hỏi phải khác nhau, không được trùng lặp.

2. **Hình thức trình bày**:
   - Trình bày hệ thống, dễ hiểu, không rườm rà.
   - Duy trì phong cách chuyên nghiệp, thân thiện và dễ tiếp cận.

---

### Định dạng yêu cầu:
- **Câu hỏi:** Bắt đầu bằng "Câu hỏi: ".
- **Câu trả lời:** Bắt đầu bằng "Trả lời: ".
- Mỗi cặp câu hỏi-trả lời được phân tách bằng một dòng trống.

---

### Ví dụ mẫu:
Câu hỏi: U xơ tử cung là gì và ảnh hưởng như thế nào đến sức khỏe phụ nữ?

Trả lời: U xơ tử cung là khối u lành tính, gây đau và rối loạn kinh nguyệt, có thể ảnh hưởng đến khả năng sinh sản.

---

Bây giờ, hãy tạo **nhiều cặp câu hỏi và câu trả lời** (nhiều nhất 10 cặp) dựa trên văn bản đã cung cấp. Đảm bảo rằng mỗi câu trả lời không chứa nội dung của câu hỏi tiếp theo và không có câu hỏi nào trùng nhau.
"""

# Khởi tạo client với API Key và Base URL của DeepSeek
client = OpenAI(api_key="dskey", base_url="https://api.deepseek.com")

# Danh sách lưu kết quả Q&A
output_data = []

# Duyệt qua từng bài viết trong DataFrame test (5 bài đầu)
for idx, row in df_test.iterrows():
    article = row["content"]
    link = row["link"]
    prompt = prompt_template.format(context=article)
    
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that creates Q&A pairs from content."},
            {"role": "user", "content": prompt}
        ],
        stream=False
    )
    
    output = response.choices[0].message.content.strip()
    
    # Giả sử các cặp Q&A được phân tách bởi dòng trống
    qa_pairs = [pair.strip() for pair in output.split("\n\n") if pair.strip()]
    
    for pair in qa_pairs:
        question, answer = "", ""
        for line in pair.split("\n"):
            if line.startswith("Câu hỏi:"):
                question = line.replace("Câu hỏi:", "").strip()
            elif line.startswith("Trả lời:"):
                answer = line.replace("Trả lời:", "").strip()
        if question and answer:
            output_data.append({
                "article_id": idx,
                "link": link,
                "question": question,
                "answer": answer
            })
    
    # Nghỉ một chút giữa các lần gọi API
    time.sleep(1)

# Tạo DataFrame chứa các cặp Q&A
df_output = pd.DataFrame(output_data)

# Lưu DataFrame ra file CSV
df_output.to_csv("qa_pairs_test.csv", index=False, encoding="utf-8-sig")

print("Đã lưu file CSV 'qa_pairs_test.csv' chứa Q&A cùng với link của bài viết gốc.")


Đã lưu file CSV 'qa_pairs_test.csv' chứa Q&A cùng với link của bài viết gốc.
